# Tutorial — Instalação e carregamento das interfaces

Um guia **curto e focado**: só (1) **instalar** a biblioteca `yggdrasil` e (2) **abrir as duas interfaces interativas** de risco de crédito —

- **`TreeSegmenterUI`** — segmentação por **árvore** (folhas como grupos de risco);
- **`ModelSegmenterUI`** — **workbench de modelo** (variáveis, ratings, métricas).

Sem análise nem modelagem aqui — apenas o mínimo para as UIs abrirem. As duas atendem **classificação (PD)** e **regressão (LGD)** pelo mesmo `task_type`.

## 1. Instalação

As interfaces usam **ipywidgets**, que vem no extra **`[ui]`** — sempre instale com esse extra. Escolha **uma** das formas:

```bash
# a) do PyPI (versão publicada) — recomendado
pip install "yggdrasil-project[ui]"

# b) da última versão da branch main (GitHub)
pip install "yggdrasil-project[ui] @ git+https://github.com/richardguilhermeds/Yggdrasil-Project.git"

# c) modo editável, a partir do repositório já clonado (desenvolvimento)
pip install -e ".[ui]"
```

> **Nomes:** o pacote de **instalação** é `yggdrasil-project`; o de **import** é `yggdrasil` (`from yggdrasil... import ...`).
>
> **Databricks:** instale como *cluster library* (PyPI `yggdrasil-project[ui]`) ou rode direto de um **Repo** (a raiz do repositório já entra no `sys.path`).

Se preferir **rodar do repositório sem instalar** (ex.: clonou e abriu o notebook), execute a célula abaixo — ela coloca a raiz do repo no `sys.path`. Já instalou pela seção 1? Pode pular.

In [ ]:
# --- OPCIONAL: rodar do REPOSITÓRIO sem `pip install` --------------------------
# Torna `yggdrasil` importável achando a raiz do repo (a pasta com `yggdrasil/`)
# e inserindo-a no sys.path. Cobre Jupyter/VS Code local e Databricks Repos. Se o
# pacote já estiver instalado (seção 1), esta célula é inócua — pode pular.
import sys
from pathlib import Path


def _find_yggdrasil_root():
    _anchors = []
    for _n in ("__vsc_ipynb_file__", "__file__", "__session__"):
        _v = globals().get(_n)
        if _v:
            _anchors.append(Path(str(_v)))
    _anchors.append(Path.cwd())
    _anchors += [Path(_p) for _p in sys.path if _p not in ("", ".")]
    for _a in _anchors:
        try:
            _a = _a.resolve()
        except Exception:
            continue
        for _b in (_a, *_a.parents):
            if (_b / "yggdrasil" / "__init__.py").is_file():
                return _b
    try:  # fallback Databricks: caminho do próprio notebook
        _nbp = (dbutils.notebook.entry_point.getDbutils()  # noqa: F821
                .notebook().getContext().notebookPath().get())
        for _pref in ("/Workspace", ""):
            for _b in Path(_pref + _nbp).parents:
                if (_b / "yggdrasil" / "__init__.py").is_file():
                    return _b
    except Exception:
        pass
    return None


_ygg_root = _find_yggdrasil_root()
if _ygg_root and str(_ygg_root) not in sys.path:
    sys.path.insert(0, str(_ygg_root))


## 2. Dados de exemplo

As duas interfaces recebem **um DataFrame do pandas**. Geramos um sintético só para elas terem o que carregar.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

# Dados MÍNIMOS só para as interfaces terem o que carregar (sintético, sem fonte
# externa). O contrato é o mesmo das duas UIs: features + `target` (alvo) +
# `amostra` (DES/OOT…) + `dt_ref` (safra, p/ os gráficos no tempo).
rng = np.random.default_rng(42)
n = 3000
meses = pd.date_range("2023-01-01", periods=12, freq="MS")

df = pd.DataFrame({
    "feat_score": rng.beta(2.5, 3, n) * 1.4 + 0.3,     # feature numérica
    "feat_garantia": rng.choice(["A", "B", "C", "D"], n),  # feature categórica
    "dt_ref": rng.choice(meses, size=n),               # safra (data de referência)
})
risco = np.clip(0.1 + 0.4 * (df["feat_score"] - 0.5), 0.01, 0.95)
df["target"] = (rng.uniform(0, 1, n) < risco).astype(int)         # alvo binário (PD)
df["amostra"] = np.where(df["dt_ref"] >= meses[8], "OOT", "DES")  # DES (dev) / OOT (out-of-time)

df.head()


## 3. Interface da **árvore** — `TreeSegmenterUI`

Importe, instancie com o DataFrame e deixe o objeto como **última linha** da célula para exibir a interface.

In [ ]:
from yggdrasil.credit_risk.tree import TreeSegmenterUI

ui_arvore = TreeSegmenterUI(
    df,
    target="target",           # coluna do alvo
    task_type="classification",  # "classification" (PD) ou "regression" (LGD)
    sample_col="amostra",      # coluna de amostra (DES/OOT…)
    ref_sample="DES",          # amostra de referência
    date_col="dt_ref",         # coluna de safra (análises no tempo)
    problem_label="PD",        # rótulo do alvo nos gráficos (opcional)
)

ui_arvore   # exibe a interface (alternativas: ui_arvore.display() ou display(ui_arvore.panel))


## 4. Interface do **model segmenter** — `ModelSegmenterUI`

Mesmo contrato de dados, mesma forma de carregar — só muda a classe importada.

In [ ]:
from yggdrasil.credit_risk.model import ModelSegmenterUI

ui_model = ModelSegmenterUI(
    df,
    target="target",
    task_type="classification",  # mesmo contrato: troque p/ "regression" (LGD)
    sample_col="amostra",
    ref_sample="DES",
    date_col="dt_ref",
    problem_label="PD",
)

ui_model    # exibe o workbench (alternativas: ui_model.display() ou display(ui_model.panel))


---

**Pronto.** As duas UIs foram carregadas a partir do mesmo `df`.

- **Exibir:** deixe `ui_arvore` / `ui_model` como última linha, ou chame `.display()`, ou `display(ui.panel)`.
- **Contrato comum:** `target` (alvo), `sample_col` (DES/OOT…), `date_col` (safra) e as demais colunas viram features.
- **Regressão (LGD):** troque `task_type="regression"` (alvo contínuo).

Para o passo a passo completo de cada uma, veja `04_tutorial_tree_segmenter.ipynb` e `06_tutorial_model_segmenter.ipynb`.